# Implementing Foundational ML Algorithms from scratch


----


## **KNN & LSH** (K-Nearest Neighbors & Locality-Sensitive Hashing)

#### Introduction:

In this notebook, I’m extending my collection of machine learning algorithms implemented from scratch by adding the **K-Nearest Neighbors (KNN)** classifier and enhancing it with **Locality Sensitive Hashing (LSH)** for efficient approximate search.  
KNN is one of the simplest yet powerful algorithms in machine learning, widely used for classification, recommendation, and information retrieval. LSH complements KNN by drastically reducing search time in high-dimensional spaces, making it scalable for larger datasets.

The focus here is on a pure NumPy implementation, without relying on external ML frameworks. This helps deepen the understanding of the algorithmic foundations behind document retrieval and similarity search.

#### Objectives:

- Implement document retrieval and classification using KNN and LSH:

    1- Preprocess and tokenize short tweet-like messages labeled into 4 categories (tech, sports, food, travel).

    2- Build document embeddings using simple averaging of word vectors.

    3- Implement KNN from scratch for classification and nearest-neighbor search.

    4- Evaluate performance in terms of accuracy and retrieval quality.

    5- Implement LSH from scratch for approximate nearest neighbor search.

    6- Compare the performance and recall of LSH vs. brute-force KNN.

-----


-----


## 1- Preprocess and tokenize short tweet-like messages labeled into 4 categories (tech, sports, food, travel).

### 1.1- Setup and export JSON dataset

In [1]:
from pathlib import Path
import json
from collections import Counter

In [19]:
# Import first tiny dataset
DATA_DIR = Path("../data")
DATA_PATH = DATA_DIR / "texts_5labels.json"

# existence & size checks
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at: {DATA_PATH.resolve()} "
                            f"(tip: confirm your working directory and relative path)")
    
size = DATA_PATH.stat().st_size
if size == 0:
    raise ValueError(f"Dataset file is empty: {DATA_PATH.resolve()}")

In [17]:
with DATA_PATH.open("r", encoding="utf-8") as f:
    data = json.load(f)        

In [18]:
texts = data["texts"]
labels = data["labels"]

In [13]:
assert len(texts) == len(labels) == 24, "Dataset should have 24 texts and 24 labels."
counts = Counter(labels)

In [14]:
print("Counts per label:", dict(counts))
print("First 3 samples:")
for i in range(3):
    print(f"[{i}] ({labels[i]}) -> {texts[i]}")

Counts per label: {'tech': 6, 'sports': 6, 'food': 6, 'travel': 6}
First 3 samples:
[0] (tech) -> AI models from scratch are fun to build with numpy
[1] (tech) -> Training neural nets takes time but it’s rewarding
[2] (tech) -> Debugging Python code late at night hits different


## 2- Build document embeddings using simple averaging of word vectors.

In [28]:
import re
from typing import List
import numpy as np

### 2.1- Canonicalize and Tokenize functions

In [22]:
# Normalize quotes, apostrophes and dashes to ASCII
def _normalize_punctuation(s: str) -> str:
    s = s.replace("’", "'").replace("‘", "'")
    s = s.replace("“", '"').replace("”", '"')
    s = s.replace("—", "-").replace("–", "-")
    return s

In [24]:
# Core cleaner function:
# - Lowercase (optional)
# - Strip URLs
# - Handle @mentions and #hashtags (keep the token text, drop the symbol)
# - Keep letters/numbers/apostrophes/hyphens inside words

def clean_text(
    s: str,
    *,
    lower: bool = True,
    keep_hashtag_text: bool = True,
    keep_mention_text: bool = True,
) -> str:
    
    # Apply normalize punctuation
    s = _normalize_punctuation(s)

    # Remove URLs
    s = re.sub(r"https?://\S+|www\.\S+", " ", s)

    # Mentions: @user -> "user" (if keep_mention_text) else removed
    if keep_mention_text:
        s = re.sub(r"@([A-Za-z0-9_]+)", r" \1 ", s)
    else:
        s = re.sub(r"@[A-Za-z0-9_]+", " ", s)

    # Hashtags: #topic -> "topic" (if keep_hashtag_text) else removed
    if keep_hashtag_text:
        s = re.sub(r"#([A-Za-z0-9_]+)", r" \1 ", s)
    else:
        s = re.sub(r"#[A-Za-z0-9_]+", " ", s)

    # Keep letters, digits, apostrophes, hyphens. Turn other punctuation into spaces
    # (allow apostrophes/hyphens inside words like: it's, state-of-the-art)
    s = re.sub(r"[^A-Za-z0-9'\-]+", " ", s)

    # Collapse multiple spaces
    s = re.sub(r"\s+", " ", s).strip()

    if lower:
        s = s.lower()

    return s

# simple tokenizer
def tokenize(s: str) -> str:
    return s.split()

In [25]:
# Apply cleaner + tokenizer
tokenized_texts = [tokenize(clean_text(t)) for t in texts]

In [26]:
# Quick preview
for i in range(3):
    print(f"[{i}] -> {tokenized_texts[i]}")

[0] -> ['ai', 'models', 'from', 'scratch', 'are', 'fun', 'to', 'build', 'with', 'numpy']
[1] -> ['training', 'neural', 'nets', 'takes', 'time', 'but', "it's", 'rewarding']
[2] -> ['debugging', 'python', 'code', 'late', 'at', 'night', 'hits', 'different']


### 2.2- Quick EDA

In [35]:
# lengths & vocab size
lengths = np.array([len(toks) for toks in tokenized_texts], dtype=int)
vocab = Counter(tok for toks in tokenized_texts for tok in toks)

In [36]:
print(f"Docs: {len(tokenized_texts)} | Vocab size: {len(vocab)}")
print(f"Per-doc tokens — min:{lengths.min()} max:{lengths.max()} mean:{lengths.mean():.2f} median:{np.median(lengths):.1f}")

# Top tokens (optional, small peek)
for tok, freq in vocab.most_common(10):
    print(f"{tok:>12s} : {freq}")

Docs: 24 | Vocab size: 160
Per-doc tokens — min:6 max:13 mean:8.75 median:8.5
         the : 9
          my : 5
         and : 5
           a : 5
          is : 5
          to : 4
        with : 4
        from : 3
          at : 3
         was : 3


We notice the presence of stopwords, however we don’t need to remove them right now because we will use TF-IDF weighting, which will naturally push common words (the, and, to, is…) toward near-zero weight. That keeps the pipeline simple and still reduces noise.

### 2.2- Apply TF-IDF

In [31]:
N = len(tokenized_texts)

# c_{t,d}: list of Counters, one per document
doc_counts = [Counter(toks) for toks in tokenized_texts]

# df_t: in how many documents each token appears (set to avoid double-counting)
df = Counter()
for toks in tokenized_texts:
    df.update(set(toks))

print(f"Docs: {N} | Unique tokens: {len(df)}")

Docs: 24 | Unique tokens: 160


In [32]:
# peek at most/least common by df, and one doc's counts

# Top 10 tokens by document frequency
print("Top by df:", df.most_common(10))

# Bottom 10 tokens by document frequency
least_df = sorted(df.items(), key=lambda kv: kv[1])[:10]
print("Least by df:", least_df)

# Example: per-doc counts for doc 0
print("Doc 0 counts:", doc_counts[0].most_common())

Top by df: [('the', 6), ('my', 5), ('and', 5), ('a', 5), ('is', 5), ('with', 4), ('to', 4), ('from', 3), ('at', 3), ('was', 3)]
Least by df: [('ai', 1), ('are', 1), ('build', 1), ('models', 1), ('scratch', 1), ('numpy', 1), ('fun', 1), ('but', 1), ('neural', 1), ('nets', 1)]
Doc 0 counts: [('ai', 1), ('models', 1), ('from', 1), ('scratch', 1), ('are', 1), ('fun', 1), ('to', 1), ('build', 1), ('with', 1), ('numpy', 1)]


In [33]:
import math

In [34]:
# Smoothed IDF (using df from step 3.1 and N = number of docs)
idf = {tok: math.log(1.0 + (N / (df_t + 1.0))) for tok, df_t in df.items()}

# Tiny diagnostics: lowest (most common) and highest (rarest) IDF tokens
lowest_idf = sorted(idf.items(), key=lambda kv: kv[1])[:10]
highest_idf = sorted(idf.items(), key=lambda kv: kv[1], reverse=True)[:10]

print("Lowest IDF (most common):", lowest_idf)
print("Highest IDF (rarest):", highest_idf)


Lowest IDF (most common): [('the', 1.488077055429833), ('my', 1.6094379124341003), ('and', 1.6094379124341003), ('a', 1.6094379124341003), ('is', 1.6094379124341003), ('with', 1.7578579175523736), ('to', 1.7578579175523736), ('from', 1.9459101490553132), ('at', 1.9459101490553132), ('was', 1.9459101490553132)]
Highest IDF (rarest): [('ai', 2.5649493574615367), ('are', 2.5649493574615367), ('build', 2.5649493574615367), ('models', 2.5649493574615367), ('scratch', 2.5649493574615367), ('numpy', 2.5649493574615367), ('fun', 2.5649493574615367), ('but', 2.5649493574615367), ('neural', 2.5649493574615367), ('nets', 2.5649493574615367)]


In [38]:
# Per-doc TF and TF-IDF weights

def tf_sublinear(c: int) -> float:
    return 0.0 if c <= 0 else 1.0 + math.log(c)

# doc_counts from previous steps
doc_tf = [{t: tf_sublinear(c) for t, c in d.items()} for d in doc_counts]
doc_tfidf = [{t: tf.get(t, 0.0) * idf.get(t, 0.0) for t in d} for tf, d in zip(doc_tf, doc_counts)]


In [39]:
# Diagnostics: check which tokens carry the most/least weight in a sample doc
d0 = 0
top_tfidf = sorted(doc_tfidf[d0].items(), key=lambda kv: kv[1], reverse=True)[:10]
low_tfidf = sorted(doc_tfidf[d0].items(), key=lambda kv: kv[1])[:10]

print("Doc 0 top by TF-IDF:", top_tfidf)
print("Doc 0 low by TF-IDF:", low_tfidf)
print("Sum of TF-IDF weights (doc 0):", sum(doc_tfidf[d0].values()))


Doc 0 top by TF-IDF: [('ai', 2.5649493574615367), ('models', 2.5649493574615367), ('scratch', 2.5649493574615367), ('are', 2.5649493574615367), ('fun', 2.5649493574615367), ('build', 2.5649493574615367), ('numpy', 2.5649493574615367), ('from', 1.9459101490553132), ('to', 1.7578579175523736), ('with', 1.7578579175523736)]
Doc 0 low by TF-IDF: [('to', 1.7578579175523736), ('with', 1.7578579175523736), ('from', 1.9459101490553132), ('ai', 2.5649493574615367), ('models', 2.5649493574615367), ('scratch', 2.5649493574615367), ('are', 2.5649493574615367), ('fun', 2.5649493574615367), ('build', 2.5649493574615367), ('numpy', 2.5649493574615367)]
Sum of TF-IDF weights (doc 0): 23.41627148639082


Import the positive and negative tweets, then initialize the raw and test sets

In [5]:
all_positive_tweets = twitter_samples.strings('positive_tweets.json')
all_negative_tweets = twitter_samples.strings('negative_tweets.json')

# split the data
test_pos = all_positive_tweets[4000:]
train_pos = all_positive_tweets[:4000]
test_neg = all_negative_tweets[4000:]
train_neg = all_negative_tweets[:4000]

train_x = train_pos + train_neg
test_x = test_pos + test_neg

train_y = np.append(np.ones(len(train_pos)), np.zeros(len(train_neg)))
test_y = np.append(np.ones(len(test_pos)), np.zeros(len(test_neg)))

In [9]:
from pathlib import Path
import sys

# project root containing /tools (local tools and libs) and /notebooks
ROOT = Path.cwd().parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [10]:
from tools.utils_NLP import process_tweet, lookup

These two imported helper functions came from the NLP Specialization Course from Deeplearning.ai (Credits).

### `process_tweet` Function

This function takes a raw tweet as input and returns a cleaned, tokenized, and stemmed list of words. It is adapted from the DeepLearning.AI NLP Specialization course.

#### Input:
- `tweet`: A string containing a tweet.

#### Processing Steps:
1. Remove unwanted text patterns:
   - Stock tickers (e.g., `$GE`)
   - Retweet indicators (`RT`)
   - Hyperlinks (e.g., `http://...`)
   - Hashtag symbols (`#`, but keeps the word)

2. Tokenize the tweet:
   - Uses `TweetTokenizer` from NLTK.
   - Converts text to lowercase.
   - Removes Twitter handles.
   - Normalizes elongated words (e.g., "cooool" → "cool").

3. Clean tokens:
   - Removes stopwords using NLTK's English stopword list.
   - Removes punctuation.
   - Applies stemming using the Porter Stemmer.

#### Output:
- `tweets_clean`: A list of cleaned and stemmed words from the original tweet.

#### Example Use Case:
This function is useful in preprocessing text for NLP tasks such as sentiment analysis or topic classification.


In [11]:
custom_tweet = "RT @Twitter @chapagain Hello There! Have a great day. :) #good #morning http://chapagain.com.np"

# print cleaned tweet
print(process_tweet(custom_tweet))

['hello', 'great', 'day', ':)', 'good', 'morn']


### `count_tweets` Function

This function builds a frequency dictionary that maps each `(word, sentiment)` pair to the number of times it appears in a given list of tweets.

#### Inputs:
- `result`: A dictionary used to store the frequency of each `(word, sentiment)` pair.
- `tweets`: A list of tweet texts.
- `ys`: A list of sentiment labels (0 for negative, 1 for positive), corresponding to each tweet.

#### Process:
- For each tweet and its corresponding sentiment:
  - The tweet is preprocessed using the `process_tweet()` function.
  - Each word in the processed tweet is paired with the sentiment label.
  - The `(word, sentiment)` pair is added to the dictionary:
    - If it already exists, its count is incremented.
    - If it's new, it is added with a count of 1.

#### Output:
- Returns the updated `result` dictionary containing the frequency of each `(word, sentiment)` pair.
> Note: The `process_tweet()` function is expected to clean and tokenize the tweet (e.g., remove punctuation, lowercase, remove stopwords, etc.).


In [12]:
def count_tweets(result, tweets, ys):
    '''
    Input:
        result: a dictionary that will be used to map each pair to its frequency
        tweets: a list of tweets
        ys: a list corresponding to the sentiment of each tweet (either 0 or 1)
    Output:
        result: a dictionary mapping each pair to its frequency
    '''
    for y, tweet in zip(ys, tweets):
        for word in process_tweet(tweet):
            # define the key, which is the word and label tuple
            pair = (word,y)
            
            # if the key exists in the dictionary, increment the count
            if pair in result:
                result[pair] += 1

            # else, if the key is new, add it to the dictionary and set the count to 1
            else:
                result[pair] = 1
    
    return result

In [13]:
# Test
result = {}
tweets = ['i am happy', 'i am tricked', 'i am sad', 'i am tired', 'i am very tired']
ys = [1, 0, 0, 0, 0]
count_tweets(result, tweets, ys)

{('happi', 1): 1, ('trick', 0): 1, ('sad', 0): 1, ('tire', 0): 2}

----

## 2- Implement the Naive Bayes algorithm from scratch

Naive Bayes is a simple and fast algorithm commonly used for sentiment analysis. Training involves estimating the probability of each class and computing word likelihoods based on frequency.

### Class Probabilities (Priors)

We start by computing the prior probability of each class. The proportion of positive and negative tweets:

$$ P(D_{pos}) = \frac{D_{pos}}{D}, \quad P(D_{neg}) = \frac{D_{neg}}{D} $$

Where:
- $D$ is the total number of tweets
- $D_{pos}$ and $D_{neg}$ are the counts of positive and negative tweets respectively

The **logprior** helps simplify calculations:

$$ \text{logprior} = \log \left( \frac{P(D_{pos})}{P(D_{neg})} \right) = \log(D_{pos}) - \log(D_{neg}) $$

### Word Likelihoods

For each word in the vocabulary, we compute its likelihood given a class (positive or negative):

$$ P(w \mid pos) = \frac{freq_{pos} + 1}{N_{pos} + V} $$
$$ P(w \mid neg) = \frac{freq_{neg} + 1}{N_{neg} + V} $$

Where:
- `freq_pos`, `freq_neg`: how often the word appears in positive/negative tweets
- `N_pos`, `N_neg`: total word counts in positive/negative tweets
- `V`: vocabulary size (number of unique words)
- The `+1` is additive smoothing to handle unseen words




### Log Likelihood

To compare how strongly a word supports the positive vs. negative class, we compute its **log likelihood**:

$$ \text{loglikelihood} = \log \left( \frac{P(w \mid pos)}{P(w \mid neg)} \right) $$

This score reflects the strength and direction of association between a word and the sentiment class.

### Frequency Dictionary (`freqs`)

Before training, we build a frequency dictionary using the `count_tweets` function:

- The dictionary `freqs` maps `(word, label)` pairs to their count.
- It allows efficient lookup of how often each word appears in each class.
- We’ll use this dictionary throughout the model training process.

In [14]:
# Build the freqs dictionary
freqs = count_tweets({}, train_x, train_y)

In [15]:
def train_naive_bayes(freqs, train_x, train_y):
    '''
    Input:
        freqs: dictionary from (word, label) to how often the word appears
        train_x: a list of tweets
        train_y: a list of labels corresponding to the tweets (0,1)
    Output:
        logprior: the log prior. (equation 3 above)
        loglikelihood: the log likelihood of you Naive bayes equation. (equation 6 above)
    '''
    loglikelihood = {}
    logprior = 0

    # calculate V, the number of unique words in the vocabulary
    vocab = list(set([key[0] for key in freqs.keys()]))
    V = len(vocab)

    # calculate N_pos, N_neg
    N_pos = N_neg = 0
    for pair in freqs.keys():
        
        if pair[1] > 0:
            N_pos += freqs[pair]
        else:
            N_neg += freqs[pair]
            
    # Calculate D, the number of documents
    D = len(train_x)

    # Calculate D_pos, the number of positive documents
    D_pos = sum(train_y)

    # Calculate D_neg, the number of negative documents
    D_neg = D - D_pos

    # Calculate logprior
    logprior = np.log(D_pos) - np.log(D_neg)
    
    # For each word in the vocabulary...
    for word in vocab:
        # get the positive and negative frequency of the word
        freq_pos = freqs.get((word,1),0)
        freq_neg = freqs.get((word,0),0)

        # calculate the probability that each word is positive, and negative
        p_w_pos = (freq_pos + 1)/(N_pos + V)
        p_w_neg = (freq_neg + 1)/(N_neg + V)

        # calculate the log likelihood of the word
        loglikelihood[word] = np.log(p_w_pos) - np.log(p_w_neg)

    return logprior, loglikelihood

In [16]:
logprior, loglikelihood = train_naive_bayes(freqs, train_x, train_y)
print(logprior)
print(len(loglikelihood))

0.0
9143


### Training Output Summary
After training the Naive Bayes model:
- **logprior = 0.0**: This indicates that the number of positive and negative tweets in the training set is equal, resulting in a neutral prior (i.e., no class is favored a priori).
- **loglikelihood contains 9,143 words**: This is the size of the vocabulary learned from the training data. Each word has an associated log likelihood score indicating how strongly it supports a positive vs. negative sentiment.

These values will be used during prediction to score new tweets based on the sum of their word log likelihoods and the logprior.


----

## 3- Make predictions on labeled data.

With the `logprior` and `loglikelihood` computed during training, we can now test the model by making predictions on new tweets.

### Task: `naive_bayes_predict`

Implement a function that predicts the sentiment of a tweet using:

$$ p = \text{logprior} + \sum_{i=1}^{N} \text{loglikelihood}_i $$

**Instructions:**
- Input: a tweet, the `logprior`, and the `loglikelihood` dictionary
- For each word in the processed tweet:
  - If it exists in `loglikelihood`, add its score to the total
- Add the `logprior` to the final sum
- Output: the overall sentiment score (positive if > 0, negative if < 0)

### Note

In our training set, classes are balanced (4000 positive and 4000 negative tweets), so:

- The prior ratio is 1 → `logprior = 0.0`
- In this case, the prediction depends only on the loglikelihoods.
- Still, always include `logprior`, as it becomes important when data is imbalanced.


In [17]:
def naive_bayes_predict(tweet, logprior, loglikelihood):
    '''
    Input:
        tweet: a string
        logprior: a number
        loglikelihood: a dictionary of words mapping to numbers
    Output:
        p: the sum of all the logliklihoods of each word in the tweet (if found in the dictionary) + logprior (a number)

    '''
    # process the tweet to get a list of words
    word_l = process_tweet(tweet)

    # initialize probability to zero
    p = 0

    # add the logprior
    p += logprior

    for word in word_l:
        # check if the word exists in the loglikelihood dictionary
        if word in loglikelihood:
            # add the log likelihood of that word to the probability
            p += loglikelihood[word]

    return p

In [18]:
# Test 1
my_tweet = 'She smiled :)'
p = naive_bayes_predict(my_tweet, logprior, loglikelihood)
print('The expected output is', p)

The expected output is 8.434581042573853


----

## 4- Evaluate the performance using accuracy as a metric

In [19]:
def test_naive_bayes(test_x, test_y, logprior, loglikelihood, naive_bayes_predict=naive_bayes_predict):
    """
    Input:
        test_x: A list of tweets
        test_y: the corresponding labels for the list of tweets
        logprior: the logprior
        loglikelihood: a dictionary with the loglikelihoods for each word
    Output:
        accuracy: (# of tweets classified correctly)/(total # of tweets)
    """
    accuracy = 0  # return this properly

    y_hats = []
    for tweet in test_x:
        if naive_bayes_predict(tweet, logprior, loglikelihood) > 0:
            y_hat_i = 1
        else:
            y_hat_i = 0
        y_hats.append(y_hat_i)

    # error is the average of the absolute values of the differences between y_hats and test_y
    error = np.mean(np.abs(y_hats - test_y))

    accuracy = 1 - error

    return accuracy

In [20]:
print("Naive Bayes accuracy = %0.4f" %
      (test_naive_bayes(test_x, test_y, logprior, loglikelihood)))

Naive Bayes accuracy = 0.9955


### Model Evaluation Result

The Naive Bayes classifier achieved an accuracy of **0.9955** on the test set.

This means the model correctly classified ~99.5% of the tweets, which is an excellent result, especially given that the implementation is from scratch and based on simple statistical assumptions.

> Keep in mind that such high accuracy could be influenced by the balance and simplicity of the dataset (preprocessed Twitter sentiment samples). For real-world applications, further validation (e.g., using precision, recall, and testing on more diverse data) would be recommended.


In [21]:
# Test the function
for tweet in ['I am happy', 'I am bad', 'this movie should have been great.', 'great', 'great great', 'great great great', 'great great great great']:
    p = naive_bayes_predict(tweet, logprior, loglikelihood)
    print(f'{tweet} -> {p:.2f}')

I am happy -> 2.13
I am bad -> -1.31
this movie should have been great. -> 2.11
great -> 2.13
great great -> 4.25
great great great -> 6.38
great great great great -> 8.50


## 🏁 Conclusion

In this notebook, we successfully implemented the **Naive Bayes classifier from scratch** for binary sentiment analysis on Twitter data.

### Key Takeaways:
- We used NLTK's `twitter_samples` and `stopwords` corpora to build and clean the dataset.
- A custom `process_tweet` function helped normalize and tokenize the tweets for modeling.
- We built a `count_tweets` frequency dictionary to capture the relationship between words and sentiment labels.
- The Naive Bayes model was trained using log priors and log likelihoods, enabling efficient probabilistic predictions.
- Our classifier achieved an impressive **accuracy of 0.9955** on the test set, correctly classifying ~99.5% of the tweets.

### Final Test Observations:
- The predictions aligned well with expectations, positive phrases (e.g., `"I am happy"` or repeated `"great"`) produced strong positive scores.
- Negative sentiment (e.g., `"I am bad"`) resulted in negative scores as expected.
- Repetition of a positively weighted word increased the prediction score linearly, demonstrating the additive nature of log likelihoods.

> While the model performed exceptionally on this dataset, real-world applications may require additional validation using more diverse and imbalanced data.

This project demonstrates how probabilistic models like Naive Bayes can be both **interpretable and effective**, especially when built with a deep understanding of the underlying mechanics.



### Limitations of Naive Bayes

Despite **its simplicity and strong baseline performance**, Naive Bayes comes with important limitations:

- **Independence Assumption**: It assumes that words are conditionally independent given the class label, which is rarely true in natural language (e.g., "not good" vs. "good").
- **Lack of Context**: The model doesn't capture the order or context of words, meaning it treats all words equally regardless of their position or relationship.
- **Vulnerability to Adversarial Inputs**: Because it's based on word counts and additive scoring, it's easy to manipulate predictions by repeating highly weighted words.

These factors make Naive Bayes less suitable for tasks requiring deeper language understanding, where more sophisticated models may perform better.


----